In [1]:
import os
import glob
import pandas as pd
import kagglehub

# ==========================================
# 1. Loading the Dataset (Parquet Format)
# ==========================================

path = kagglehub.dataset_download("mohammedalsubaie/king-khalid-international-airport-flights-dataset")

parquet_files = glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)

df = pd.read_parquet(parquet_files[0])

print(df.columns.tolist())

['flight_number', 'aircraft.model', 'aircraft.reg', 'aircraft.modeS', 'airline.name', 'airline.iata', 'airline.icao', 'status', 'flight_type', 'codeshareStatus', 'isCargo', 'callSign', 'origin_airport_name', 'origin_airport_icao', 'origin_airport_iata', 'movement.terminal', 'movement.quality', 'destination_airport_icao', 'destination_airport_iata', 'destination_airport_name', 'movement.airport.timeZone', 'movement.scheduledTime.utc', 'movement.scheduledTime.local']


In [2]:
print(df.head())


df.info()

  flight_number   aircraft.model aircraft.reg aircraft.modeS  \
0        PF 769      Airbus A320         None           None   
1        XY 333  Airbus A320 NEO      HZ-NS35         710DB9   
2        QP 568       Boeing 737         None           None   
3        F3 161      Airbus A320         None           None   
4        KL 423  Airbus A330-300         None           None   

        airline.name airline.iata airline.icao   status flight_type  \
0           Air Sial           PF         None  Unknown   departure   
1             flynas           XY          KNE  Unknown   departure   
2  Starlight Airline           QP          SLT  Unknown   departure   
3           flyadeal           F3          FAD  Unknown   departure   
4                KLM           KL          KLM  Unknown   departure   

  codeshareStatus  ...  origin_airport_icao origin_airport_iata  \
0         Unknown  ...                 OERK                 RUH   
1      IsOperator  ...                 OERK           

In [3]:
import numpy as np


# ==========================================
# 2. Data Cleaning
# ==========================================

df['movement.scheduledTime.local'] = pd.to_datetime(df['movement.scheduledTime.local'], errors='coerce')
df['movement.scheduledTime.utc'] = pd.to_datetime(df['movement.scheduledTime.utc'], errors='coerce')
df = df.dropna(subset=['movement.scheduledTime.local']).copy()

cols_to_drop = ['aircraft.modeS', 'callSign', 'movement.quality']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df['status'] = df['status'].replace(['Unknown', 'unknown', None, np.nan], 'Unspecified')
df['codeshareStatus'] = df['codeshareStatus'].replace(['Unknown', 'unknown', None, np.nan], 'Direct Flight')
df['movement.terminal'] = df['movement.terminal'].fillna('Unknown Terminal').astype(str).str.replace('.0', '', regex=False)
df['airline.name'] = df['airline.name'].fillna('Unknown Airline').str.strip()
df['aircraft.model'] = df['aircraft.model'].fillna('Unknown Model').str.strip()
df['origin_airport_name'] = df['origin_airport_name'].fillna('Unknown Airport').str.strip()
df['destination_airport_name'] = df['destination_airport_name'].fillna('Unknown Airport').str.strip()

df = df.drop_duplicates()

# ==========================================
# 3. Dynamic Saudi Airports Identification
# ==========================================
origin_saudi_mask = df['origin_airport_icao'].str.startswith('OE', na=False)
dest_saudi_mask = df['destination_airport_icao'].str.startswith('OE', na=False)

saudi_airports_iata = set(
    df.loc[origin_saudi_mask, 'origin_airport_iata'].dropna().tolist() + 
    df.loc[dest_saudi_mask, 'destination_airport_iata'].dropna().tolist()
)

# ==========================================
# 4. Data Quality Report
# ==========================================
total_rows = len(df)
total_cols = len(df.columns)
dup_rows = df.duplicated().sum()
missing_pct = (df.isnull().sum().sum() / (total_rows * total_cols)) * 100

origin_iata = df['origin_airport_iata'].fillna('UNK')
dest_iata = df['destination_airport_iata'].fillna('UNK')
is_dom = origin_iata.isin(saudi_airports_iata) & dest_iata.isin(saudi_airports_iata)

dom_pct = (is_dom.sum() / total_rows) * 100
intl_pct = 100 - dom_pct
cargo_count = df['isCargo'].sum() if 'isCargo' in df.columns else 0

print("=" * 45)
print("          DATA QUALITY REPORT          ")
print("=" * 45)
print(f"Total Rows              : {total_rows:,}")
print(f"Total Columns           : {total_cols}")
print(f"Duplicate Rows          : {dup_rows}")
print(f"Missing Values (%)      : {missing_pct:.2f}%")
print("-" * 45)
print(f"Unique Airlines         : {df['airline.name'].nunique():,}")
print(f"Unique Origin Airports  : {df['origin_airport_name'].nunique():,}")
print(f"Unique Dest Airports    : {df['destination_airport_name'].nunique():,}")
print(f"Unique Aircraft Models  : {df['aircraft.model'].nunique():,}")
print("-" * 45)
print(f"Date Range (Min)        : {df['movement.scheduledTime.local'].min()}")
print(f"Date Range (Max)        : {df['movement.scheduledTime.local'].max()}")
print("-" * 45)
print(f"Cargo Flights Count     : {cargo_count:,}")
print(f"Domestic Flights (%)    : {dom_pct:.2f}%")
print(f"International Flights(%): {intl_pct:.2f}%")
print("=" * 45 + "\n")

# ==========================================
# 5. Vectorized Feature Engineering
# ==========================================
local_time = df['movement.scheduledTime.local']

df['Date'] = local_time.dt.date
df['Year'] = local_time.dt.year
df['Month_Number'] = local_time.dt.month
df['Month_Name'] = local_time.dt.strftime('%B')
df['Quarter'] = 'Q' + local_time.dt.quarter.astype(str)
df['Week_Number'] = local_time.dt.isocalendar().week
df['Day_Name'] = local_time.dt.strftime('%A')
df['Day_of_Week'] = local_time.dt.dayofweek
df['Is_Weekend'] = np.where(df['Day_of_Week'].isin([4, 5]), 'Yes', 'No')
df['Hour'] = local_time.dt.hour
df['Minute'] = local_time.dt.minute

# Season
month = df['Month_Number']
season_conditions = [month.isin([12, 1, 2]), month.isin([3, 4, 5]), month.isin([6, 7, 8])]
df['Season'] = np.select(season_conditions, ['Winter', 'Spring', 'Summer'], default='Autumn')

# Time Period
hour = df['Hour']
time_conditions = [
    (hour >= 0) & (hour < 4), (hour >= 4) & (hour < 8),
    (hour >= 8) & (hour < 12), (hour >= 12) & (hour < 16),
    (hour >= 16) & (hour < 20)
]
df['Time_Period'] = np.select(time_conditions, ['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening'], default='Night')

# Shift
shift_conditions = [(hour >= 0) & (hour < 8), (hour >= 8) & (hour < 16)]
df['Flight_Shift'] = np.select(shift_conditions, ['Shift A', 'Shift B'], default='Shift C')

# Routes
df['Flight_Direction'] = df['flight_type'].astype(str).str.capitalize()
df['Route_IATA'] = origin_iata + ' → ' + dest_iata
df['Route_Full_Name'] = df['origin_airport_name'] + ' → ' + df['destination_airport_name']
df['Route_Type'] = np.where(is_dom, 'Domestic', 'International')

# Aircraft Classification
model_upper = df['aircraft.model'].str.upper()

manuf_conditions = [
    model_upper.str.contains('AIRBUS|A3', regex=True), model_upper.str.contains('BOEING|B7', regex=True),
    model_upper.str.contains('EMBRAER|ERJ|E1', regex=True), model_upper.str.contains('ATR', regex=True),
    model_upper.str.contains('BOMBARDIER|CRJ', regex=True)
]
df['Aircraft_Manufacturer'] = np.select(manuf_conditions, ['Airbus', 'Boeing', 'Embraer', 'ATR', 'Bombardier'], default='Other')

family_conditions = [
    model_upper.str.contains('A318|A319|A320|A321', regex=True), model_upper.str.contains('A330', regex=True),
    model_upper.str.contains('A350', regex=True), model_upper.str.contains('A380', regex=True),
    model_upper.str.contains('737', regex=True), model_upper.str.contains('777', regex=True),
    model_upper.str.contains('787|DREAMLINER', regex=True), model_upper.str.contains('747', regex=True),
    model_upper.str.contains('ATR', regex=True), model_upper.str.contains('EMBRAER|ERJ|E1', regex=True),
    model_upper.str.contains('BOMBARDIER|CRJ', regex=True)
]
family_choices = [
    'Airbus A320 Family', 'Airbus A330 Family', 'Airbus A350 Family', 'Airbus A380 Family',
    'Boeing 737 Family', 'Boeing 777 Family', 'Boeing 787 Family', 'Boeing 747 Family',
    'ATR Family', 'Embraer E-Jet', 'Bombardier CRJ'
]
df['Aircraft_Family'] = np.select(family_conditions, family_choices, default='Other Family')
df['Is_Wide_Body'] = np.where(model_upper.str.contains('777|787|747|A330|A350|A380|330|350|380', regex=True), 'Yes', 'No')

# Airline Categories
airline_lower = df['airline.name'].str.lower()
national_carriers = 'saudi arabian|saudia|flynas|flyadeal'
gcc_carriers = 'emirates|qatar airways|gulf air|kuwait airways|etihad|flydubai|air arabia|oman air|jazeera'

airline_conditions = [
    airline_lower.str.contains(national_carriers, regex=True),
    airline_lower.str.contains(gcc_carriers, regex=True)
]
df['Airline_Category'] = np.select(airline_conditions, ['National', 'GCC / Regional'], default='International')

# Aggregations
df['Daily_Flights'] = df.groupby('Date')['flight_number'].transform('count')
df['Hourly_Flights'] = df.groupby(['Date', 'Hour'])['flight_number'].transform('count')
df['Monthly_Flights'] = df.groupby(['Year', 'Month_Number'])['flight_number'].transform('count')
df['Flights_per_Terminal'] = df.groupby('movement.terminal')['flight_number'].transform('count')
df['Terminal_Utilization_Pct'] = np.round((df['Flights_per_Terminal'] / total_rows) * 100, 2)

# Traffic Level
q25, q50, q75, q90 = df['Hourly_Flights'].quantile([0.25, 0.50, 0.75, 0.90])
traffic_conditions = [
    df['Hourly_Flights'] <= q25, df['Hourly_Flights'] <= q50,
    df['Hourly_Flights'] <= q75, df['Hourly_Flights'] <= q90
]
df['Traffic_Level'] = np.select(traffic_conditions, ['Very Low', 'Low', 'Medium', 'High'], default='Extreme')
df['Peak_Hour'] = np.where(df['Hourly_Flights'] >= q75, 'Peak', 'Non Peak')

# Ranks & Activity
df['Airline_Rank'] = df['airline.name'].map(df['airline.name'].value_counts().rank(ascending=False, method='min')).astype(int)
df['Route_Rank'] = df['Route_IATA'].map(df['Route_IATA'].value_counts().rank(ascending=False, method='min')).astype(int)
df['Destination_Rank'] = df['destination_airport_name'].map(df['destination_airport_name'].value_counts().rank(ascending=False, method='min')).astype(int)

h_min, h_max = df['Hourly_Flights'].min(), df['Hourly_Flights'].max()
d_min, d_max = df['Daily_Flights'].min(), df['Daily_Flights'].max()
norm_hourly = ((df['Hourly_Flights'] - h_min) / (h_max - h_min)) * 100 if h_max > h_min else 0
norm_daily = ((df['Daily_Flights'] - d_min) / (d_max - d_min)) * 100 if d_max > d_min else 0
df['Airport_Activity_Score'] = np.round((norm_hourly + norm_daily) / 2, 2)

# ==========================================
# 6. Convert Columns to Category
# ==========================================
cat_columns = [
    'Season', 'Time_Period', 'Flight_Shift', 'Route_Type', 
    'Airline_Category', 'Traffic_Level', 'Peak_Hour', 
    'Is_Weekend', 'Is_Wide_Body', 'Aircraft_Manufacturer', 'Aircraft_Family'
]
for col in cat_columns:
    df[col] = df[col].astype('category')

# ==========================================
# 7. Building Star Schema Architecture & Mapping Keys
# ==========================================
df['FlightKey'] = np.arange(1, len(df) + 1)
df['DateKey'] = df['movement.scheduledTime.local'].dt.strftime('%Y%m%d').astype(int)

# DimDate
dim_date = df[[
    'DateKey', 'Date', 'Year', 'Quarter', 'Month_Number', 'Month_Name', 
    'Week_Number', 'Day_Name', 'Day_of_Week', 'Is_Weekend', 'Season'
]].drop_duplicates().reset_index(drop=True)

# DimAirline
dim_airline = df[['airline.name', 'airline.iata', 'airline.icao', 'Airline_Category']].drop_duplicates(subset=['airline.name']).reset_index(drop=True)
dim_airline['AirlineKey'] = np.arange(1, len(dim_airline) + 1)

# DimAircraft
dim_aircraft = df[['aircraft.model', 'Aircraft_Manufacturer', 'Aircraft_Family', 'Is_Wide_Body']].drop_duplicates(subset=['aircraft.model']).reset_index(drop=True)
dim_aircraft['AircraftKey'] = np.arange(1, len(dim_aircraft) + 1)

# DimTerminal
dim_terminal = df[['movement.terminal', 'Flights_per_Terminal', 'Terminal_Utilization_Pct']].drop_duplicates(subset=['movement.terminal']).reset_index(drop=True)
dim_terminal['TerminalKey'] = np.arange(1, len(dim_terminal) + 1)
dim_terminal = dim_terminal.rename(columns={'movement.terminal': 'TerminalName'})

# DimAirport
airports_origin = df[['origin_airport_name', 'origin_airport_iata', 'origin_airport_icao']].rename(
    columns={'origin_airport_name': 'Airport_Name', 'origin_airport_iata': 'Airport_IATA', 'origin_airport_icao': 'Airport_ICAO'}
)
airports_dest = df[['destination_airport_name', 'destination_airport_iata', 'destination_airport_icao']].rename(
    columns={'destination_airport_name': 'Airport_Name', 'destination_airport_iata': 'Airport_IATA', 'destination_airport_icao': 'Airport_ICAO'}
)
dim_airport = pd.concat([airports_origin, airports_dest]).drop_duplicates(subset=['Airport_IATA']).dropna(subset=['Airport_IATA']).reset_index(drop=True)
dim_airport['AirportKey'] = np.arange(1, len(dim_airport) + 1)
dim_airport['Is_Saudi_Airport'] = np.where(dim_airport['Airport_IATA'].isin(saudi_airports_iata), 'Yes', 'No')

# Map Keys Back to Fact (df)
df = df.merge(dim_airline[['airline.name', 'AirlineKey']], on='airline.name', how='left')
df = df.merge(dim_aircraft[['aircraft.model', 'AircraftKey']], on='aircraft.model', how='left')
df = df.merge(dim_terminal[['TerminalName', 'TerminalKey']], left_on='movement.terminal', right_on='TerminalName', how='left')
df = df.merge(dim_airport[['Airport_IATA', 'AirportKey']].rename(columns={'AirportKey': 'OriginAirportKey'}), left_on='origin_airport_iata', right_on='Airport_IATA', how='left').drop(columns=['Airport_IATA'])
df = df.merge(dim_airport[['Airport_IATA', 'AirportKey']].rename(columns={'AirportKey': 'DestinationAirportKey'}), left_on='destination_airport_iata', right_on='Airport_IATA', how='left').drop(columns=['Airport_IATA'])

# ==========================================
# 8. Fact Table Extraction
# ==========================================
fact_ordered_columns = [
    'FlightKey', 'DateKey', 'AirlineKey', 'AircraftKey', 'TerminalKey', 'OriginAirportKey', 'DestinationAirportKey',
    'flight_number', 'status', 'flight_type', 'codeshareStatus', 'isCargo',
    'Hour', 'Minute', 'Time_Period', 'Flight_Shift',
    'Route_IATA', 'Route_Full_Name', 'Route_Type',
    'Hourly_Flights', 'Daily_Flights', 'Monthly_Flights', 'Traffic_Level', 'Peak_Hour', 
    'Airport_Activity_Score', 'Airline_Rank', 'Route_Rank', 'Destination_Rank'
]

fact_flights = df[fact_ordered_columns].copy()

# ==========================================
# 9. KPI Summary Table Export
# ==========================================
kpi_data = {
    'KPI_Name': [
        'Total Flights', 'Total Airlines', 'Total Unique Airports', 'Total Aircraft Models',
        'Cargo Flights', 'Domestic Flights Count', 'International Flights Count',
        'Busiest Airline', 'Busiest Route', 'Busiest Terminal', 'Peak Hour Range'
    ],
    'KPI_Value': [
        str(total_rows),
        str(df['airline.name'].nunique()),
        str(dim_airport['Airport_IATA'].nunique()),
        str(dim_aircraft['aircraft.model'].nunique()),
        str(cargo_count),
        str(is_dom.sum()),
        str((~is_dom).sum()),
        df['airline.name'].mode()[0],
        df['Route_IATA'].mode()[0],
        df['movement.terminal'].mode()[0],
        f"{df[df['Peak_Hour']=='Peak']['Hour'].mode()[0]}:00"
    ]
}
kpi_summary_df = pd.DataFrame(kpi_data)

# ==========================================
# 10. File Exports
# ==========================================
output_dir = "PowerBI_Star_Schema"
os.makedirs(output_dir, exist_ok=True)

df.to_csv("Airport_Analytics.csv", index=False, encoding='utf-8-sig')
fact_flights.to_csv(os.path.join(output_dir, "FactFlights.csv"), index=False, encoding='utf-8-sig')
dim_date.to_csv(os.path.join(output_dir, "DimDate.csv"), index=False, encoding='utf-8-sig')
dim_airline.to_csv(os.path.join(output_dir, "DimAirline.csv"), index=False, encoding='utf-8-sig')
dim_aircraft.to_csv(os.path.join(output_dir, "DimAircraft.csv"), index=False, encoding='utf-8-sig')
dim_terminal.to_csv(os.path.join(output_dir, "DimTerminal.csv"), index=False, encoding='utf-8-sig')
dim_airport.to_csv(os.path.join(output_dir, "DimAirport.csv"), index=False, encoding='utf-8-sig')
kpi_summary_df.to_csv(os.path.join(output_dir, "KPI_Summary.csv"), index=False, encoding='utf-8-sig')

print("All tasks completed successfully!")
print("Files exported to folder:", output_dir)

          DATA QUALITY REPORT          
Total Rows              : 153,205
Total Columns           : 20
Duplicate Rows          : 0
Missing Values (%)      : 4.24%
---------------------------------------------
Unique Airlines         : 68
Unique Origin Airports  : 1
Unique Dest Airports    : 128
Unique Aircraft Models  : 49
---------------------------------------------
Date Range (Min)        : 2025-03-15 00:01:00+03:00
Date Range (Max)        : 2025-10-10 11:45:00+03:00
---------------------------------------------
Cargo Flights Count     : 0
Domestic Flights (%)    : 51.79%
International Flights(%): 48.21%

All tasks completed successfully!
Files exported to folder: PowerBI_Star_Schema
